In [0]:
from pyspark.sql.functions import *

# Load the trust-scored dataset
df = spark.table("healthcare_facilities_cleaned")

# Verify trust scores are there
print(f"Total facilities: {df.count()}")
print(f"Has trust_score column: {'trust_score' in df.columns}")
print(f"Has combined_text column: {'combined_text' in df.columns}")

# Check combined_text is ready for embedding
print(f"Null combined_text: {df.filter(col('combined_text').isNull()).count()}")
print(f"Avg text length: {df.select(avg(length(col('combined_text')))).first()[0]:.0f} chars")

Total facilities: 10000
Has trust_score column: True
Has combined_text column: True
Null combined_text: 0
Avg text length: 616 chars


In [0]:
%pip install databricks-sdk mlflow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

# Test with one sample text
try:
    response = client.predict(
        endpoint="databricks-bge-large-en",
        inputs={"input": ["Dental clinic offering RCT and Laser Dentistry"]}
    )
    embedding = response.data[0]["embedding"]
    print(f"✅ Embedding model works!")
    print(f"Embedding dimension: {len(embedding)}")
    print(f"First 5 values: {embedding[:5]}")
except Exception as e:
    print(f"❌ Error: {str(e)[:300]}")
    print("\n💡 We'll fall back to keyword search instead")

✅ Embedding model works!
Embedding dimension: 1024
First 5 values: [-0.02496337890625, 0.0208740234375, -0.022674560546875, 0.0104522705078125, -0.034149169921875]


In [0]:
import json
import time

# ============================================================================
# EMBED ALL 10,000 FACILITIES
# ============================================================================

# Collect combined_text from all facilities
print("📥 Collecting text from all facilities...")
all_rows = df.select("name_cleaned", "combined_text").collect()
print(f"   Collected {len(all_rows)} rows")

# Batch embed - API handles ~20 texts at a time safely
BATCH_SIZE = 20
all_embeddings = []
errors = 0

print(f"\n🔄 Embedding {len(all_rows)} facilities in batches of {BATCH_SIZE}...")
print(f"   Estimated batches: {len(all_rows) // BATCH_SIZE + 1}\n")

for i in range(0, len(all_rows), BATCH_SIZE):
    batch = all_rows[i:i + BATCH_SIZE]
    texts = [row["combined_text"][:2000] for row in batch]  # Truncate to 2000 chars
    
    try:
        response = client.predict(
            endpoint="databricks-bge-large-en",
            inputs={"input": texts}
        )
        batch_embeddings = [item["embedding"] for item in response.data]
        all_embeddings.extend(batch_embeddings)
    except Exception as e:
        # On error, add zero vectors as placeholder
        errors += 1
        all_embeddings.extend([[0.0] * 1024] * len(texts))
        if errors <= 3:
            print(f"   ⚠️ Batch {i//BATCH_SIZE} failed: {str(e)[:100]}")
        time.sleep(2)  # Wait before retry
    
    # Progress update every 100 batches
    if (i // BATCH_SIZE) % 100 == 0 and i > 0:
        print(f"   ✅ Processed {i}/{len(all_rows)} ({i*100//len(all_rows)}%)")

print(f"\n✅ Embedding complete!")
print(f"   Total embeddings: {len(all_embeddings)}")
print(f"   Errors: {errors}")
print(f"   Dimension: {len(all_embeddings[0])}")

📥 Collecting text from all facilities...
   Collected 10000 rows

🔄 Embedding 10000 facilities in batches of 20...
   Estimated batches: 501

   ✅ Processed 2000/10000 (20%)
   ✅ Processed 4000/10000 (40%)
   ✅ Processed 6000/10000 (60%)
   ✅ Processed 8000/10000 (80%)

✅ Embedding complete!
   Total embeddings: 10000
   Errors: 0
   Dimension: 1024


In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()
print("✅ Vector Search client available")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
✅ Vector Search client available


In [0]:
# ============================================================================
# STEP 1: Use Existing Vector Search Endpoint
# ============================================================================

# Use the existing endpoint (workspace quota allows only 1)
endpoint_name = "healthcare_vector_search_endpoint"

try:
    # Check if endpoint exists
    endpoint = vsc.get_endpoint(endpoint_name)
    print(f"✅ Using existing endpoint '{endpoint_name}'")
    print(f"📊 Endpoint status: {endpoint.get('endpoint_status', {}).get('state', 'unknown')}")
except Exception as e:
    print(f"❌ Error accessing endpoint: {str(e)}")
    print("💡 Check if endpoint exists with: vsc.list_endpoints()")

✅ Using existing endpoint 'healthcare_vector_search_endpoint'
📊 Endpoint status: ONLINE


In [0]:
# ============================================================================
# Create a lean search table with just what Vector Search needs
# ============================================================================

from pyspark.sql import Row
from pyspark.sql.types import *

print("📦 Building search table...")

# Combine facility data + embeddings
search_data = []
for i, row in enumerate(all_rows):
    search_data.append(Row(
        id=i,
        name_cleaned=row["name_cleaned"],
        combined_text=row["combined_text"][:2000],
        embedding=all_embeddings[i]
    ))

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name_cleaned", StringType(), True),
    StructField("combined_text", StringType(), True),
    StructField("embedding", ArrayType(FloatType()), True)
])

search_df = spark.createDataFrame(search_data, schema)

# Save as Delta table
search_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare_search_index")

print(f"✅ Search table saved: {search_df.count()} rows")
print(f"   Columns: {search_df.columns}")

📦 Building search table...
✅ Search table saved: 10000 rows
   Columns: ['id', 'name_cleaned', 'combined_text', 'embedding']


In [0]:
# build-in vector store VectorSearchClient in DBR 13.3 LTS and above

from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient()

# ============================================================================
# STEP 3A: Enable Change Data Feed on the source table
# ============================================================================

source_table_name = "workspace.default.healthcare_search_index"

print("🔧 Enabling Change Data Feed on source table...")

try:
    spark.sql(f"ALTER TABLE {source_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"   ✅ CDF enabled on {source_table_name}")

except Exception as e:
    print(f"   ⚠️ {str(e)[:200]}")


# ============================================================================
# STEP 3B: Create Vector Search Index with DELTA_SYNC
# ============================================================================

endpoint_name = "healthcare_vector_search_endpoint"
index_name = "workspace.default.healthcare_vector_index"

print("\n🔄 Creating Vector Search Index...")
print("   Using DELTA_SYNC mode")
print("   This may take a few minutes...\n")

try:
    index = vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
        source_table_name=source_table_name,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_source_column="combined_text",
        embedding_model_endpoint_name="databricks-bge-large-en"
    )
    print(f"✅ Index created: {index_name}")
    print(f"📊 Index will automatically sync embeddings from the source table")
    
    
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"✅ Index already exists: {index_name}")
        print(f"   Access it with: vsc.get_index('{index_name}')")
    else:
        print(f"❌ Error: {str(e)[:400]}")
        print("\n💡 Check table properties with:")
        print(f"   DESCRIBE DETAIL {source_table_name}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔧 Enabling Change Data Feed on source table...
   ✅ CDF enabled on workspace.default.healthcare_search_index

🔄 Creating Vector Search Index...
   Using DELTA_SYNC mode
   This may take a few minutes...

✅ Index already exists: workspace.default.healthcare_vector_index
   Access it with: vsc.get_index('workspace.default.healthcare_vector_index')


In [0]:
import mlflow.deployments

# ============================================================================
# TEST: Semantic Search
# ============================================================================

# Recreate client (needed after Python restart in Cell 5)
client = mlflow.deployments.get_deploy_client("databricks")

# Get the index
index = vsc.get_index(
    endpoint_name="healthcare_vector_search_endpoint",
    index_name="workspace.default.healthcare_vector_index"
)

# Embed a test query
test_query = "Find a hospital with emergency surgery capability in Bihar"

query_embedding = client.predict(
    endpoint="databricks-bge-large-en",
    inputs={"input": [test_query]}
).data[0]["embedding"]

# Search
results = index.similarity_search(
    query_vector=query_embedding,
    num_results=5,
    columns=["id", "name_cleaned", "combined_text"]
)

print(f"🔍 Query: {test_query}\n")
print(f"📊 Top 5 Results:\n")

for i, row in enumerate(results.get("result", {}).get("data_array", [])):
    print(f"{i+1}. {row[1]}")
    print(f"   Text: {row[2][:150]}...")
    print(f"   Score: {row[3]:.4f}\n")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Query: Find a hospital with emergency surgery capability in Bihar

📊 Top 5 Results:

1. Bihar Netralaya Superspeciality Eye Hospital
   Text: Facility: Bihar Netralaya Superspeciality Eye Hospital | Type: hospital | Location: Munger, Bihar - 811201 | Description: Eye Care Center offering adv...
   Score: 0.5997

2. JK Hospital & Trauma Centre, Babrala
   Text: Facility: JK Hospital & Trauma Centre, Babrala | Type: hospital | Location: Babrala, Uttar Pradesh - 202522 | Description: No description | Specialtie...
   Score: 0.5936

3. I Health Super Specialty Clinic
   Text: Facility: I Health Super Specialty Clinic | Type: clinic | Location: Patna, Bihar - 801505 | Description: liver clinic | Specialties: familyMedicine |...
   Score: 0.5925

4. Erattupetta Medical Centre
   Text